# 04A.决策树初筛_含评分

使用基础评分 + 数据产品变量进行决策树初筛。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

import lightgbm as lgb
from sklearn.model_selection import train_test_split

In [ ]:
version_tag = "A"   # A=3A score全保留, B=3B score筛选

if version_tag == "A":
    fea_file = OUTPUT_DIR / f"3A.单变量初筛结果_score全保留_{customer_type}_{target_type}.csv"
    data_file = OUTPUT_DIR / f"3A.单变量初筛数据_score全保留_{customer_type}_{target_type}.csv"
elif version_tag == "B":
    fea_file = OUTPUT_DIR / f"3B.单变量初筛结果_score筛选_{customer_type}_{target_type}.csv"
    data_file = OUTPUT_DIR / f"3B.单变量初筛数据_score筛选_{customer_type}_{target_type}.csv"

fea_info = pd.read_csv(fea_file)
data_df = pd.read_csv(data_file).fillna(-999)

train_valid_data = data_df[data_df["flag"].isin(["train", "valid"])].copy()
oot_data = data_df[data_df["flag"] == "oot"].copy()

fea_info["prod"] = fea_info["var"].apply(get_product_prefix)

score_cols = [c for c in fea_info["var"].tolist() if str(c).startswith("score") and c in train_valid_data.columns]
used_base_score = score_cols[:1]

print("used_base_score:", used_base_score)

In [ ]:
fea_cols = fea_info[fea_info["dtype"].isin(["数值型", "类别型"])]["var"].tolist()
fea_cols = [c for c in fea_cols if c in train_valid_data.columns and not c.startswith("score")]

col_vars = fea_cols + used_base_score

cat_lst = [c for c in col_vars if str(train_valid_data[c].dtype) in ["object", "category"]]

for col in cat_lst:
    all_values = pd.concat([train_valid_data[col], oot_data[col]], axis=0).astype(str)
    categories = all_values.astype("category").cat.categories
    train_valid_data[col] = pd.Categorical(train_valid_data[col].astype(str), categories=categories).codes
    oot_data[col] = pd.Categorical(oot_data[col].astype(str), categories=categories).codes

print(len(col_vars), len(cat_lst))

In [ ]:
if used_base_score:
    bins_lst = [300, 600, 650, 700, 750, 800, 850, 900, 1000]
    base_score = used_base_score[0]
    train_valid_data["score_bins"] = train_valid_data[base_score].apply(lambda x: score_divide(x, bins_lst))
    display(model_dictionary_score(train_valid_data, var=base_score, bins=bins_lst, flagy=target, risk_direction="low"))
else:
    train_valid_data["score_bins"] = 0

In [ ]:
params = {
    "feature_fraction": 0.8,
    "bagging_fraction": 0.6,
    "bagging_freq": 1,
    "n_estimators": 50,
    "learning_rate": 0.003,
    "num_leaves": 3,
    "lambda_l1": 40,
    "lambda_l2": 100,
    "nthread": 7,
    "boosting_type": "gbdt",
    "is_unbalance": True,
    "metric": "auc",
    "objective": "binary",
    "min_data_in_leaf": 2000,
    "verbosity": -1
}

good_rands = [1]
var_info_df = pd.DataFrame()

for rand in good_rands:
    X_train, X_test, y_train, y_test = train_test_split(
        train_valid_data[col_vars],
        train_valid_data[target],
        test_size=0.3,
        random_state=rand,
        stratify=train_valid_data[["score_bins", target]]
    )
    train_set = lgb.Dataset(X_train, y_train, categorical_feature=cat_lst)
    gbm = lgb.train(params, train_set)

    imp = pd.DataFrame({
        "feature_name": gbm.feature_name(),
        "feature_importance": gbm.feature_importance(importance_type="gain"),
        "rand": rand
    })
    var_info_df = pd.concat([var_info_df, imp], axis=0)

selected_var_info = var_info_df.copy()
selected_var_info["prod"] = selected_var_info["feature_name"].apply(get_product_prefix)
selected_stat_info = selected_var_info.groupby(["prod", "feature_name"])[["feature_importance"]].mean().reset_index()
selected_stat_info.to_csv(OUTPUT_DIR / f"4{version_tag}.决策树初筛_产品阶段_含评分_{customer_type}_{target_type}.csv", index=False)

In [ ]:
prod_limit = 11  # 1 score + 10 products
selected_prod_lst = used_base_score.copy()
prod_cnt = len(selected_prod_lst)

wait_prods = (
    selected_stat_info[selected_stat_info["feature_importance"] > 0]
    .sort_values("feature_importance", ascending=False)["prod"]
    .tolist()
)

wait_index = 0
while prod_cnt < prod_limit and wait_index < len(wait_prods):
    cur_prod = wait_prods[wait_index]

    if cur_prod in selected_prod_lst:
        wait_index += 1
        continue

    if str(cur_prod).startswith("score"):
        wait_index += 1
        continue

    selected_prod_lst.append(cur_prod)
    prod_cnt += 1
    wait_index += 1

rest_col_vars = selected_stat_info[selected_stat_info["prod"].isin(selected_prod_lst)]["feature_name"].tolist()

for s in used_base_score:
    if s not in rest_col_vars:
        rest_col_vars.append(s)

print(selected_prod_lst)
print([c for c in rest_col_vars if c.startswith("score")])

In [ ]:
rest_var_info_df = pd.DataFrame()

for rand in good_rands:
    X_train, X_test, y_train, y_test = train_test_split(
        train_valid_data[rest_col_vars],
        train_valid_data[target],
        test_size=0.3,
        random_state=rand,
        stratify=train_valid_data[["score_bins", target]]
    )

    train_set = lgb.Dataset(X_train, y_train)
    gbm = lgb.train(params, train_set)

    imp = pd.DataFrame({
        "feature_name": gbm.feature_name(),
        "feature_importance": gbm.feature_importance(importance_type="gain"),
        "rand": rand
    })
    rest_var_info_df = pd.concat([rest_var_info_df, imp], axis=0)

rest_var_info_df["prod"] = rest_var_info_df["feature_name"].apply(get_product_prefix)
rest_stat_info = rest_var_info_df.groupby(["prod", "feature_name"])[["feature_importance"]].mean().reset_index()
rest_stat_info.to_csv(OUTPUT_DIR / f"4{version_tag}.决策树初筛_初筛结果_含评分_{customer_type}_{target_type}.csv", index=False)

final_vars = rest_stat_info[rest_stat_info["feature_importance"] > 0]["feature_name"].tolist()

for s in used_base_score:
    if s not in final_vars:
        final_vars.append(s)

final_train_valid_data = train_valid_data[join_keys + ["flag", target] + final_vars].copy()
final_oot_data = oot_data[join_keys + ["flag", target] + final_vars].copy()

final_train_valid_data.to_csv(OUTPUT_DIR / f"4{version_tag}.train_valid_data_含评分_{customer_type}_{target_type}.csv", index=False)
final_oot_data.to_csv(OUTPUT_DIR / f"4{version_tag}.oot_data_含评分_{customer_type}_{target_type}.csv", index=False)

print(final_train_valid_data.shape, final_oot_data.shape)